# 06 分子空间可视化：从高维 fingerprint 到二维图

这一节把 fingerprint 从高维投影到二维，帮助观察分子空间。

应用场景：检查数据覆盖范围、发现离群分子、观察性质梯度、为主动学习或实验设计提供直觉。
本节用 scikit-learn 的 PCA 做最基础的线性降维；它适合教学，但不能替代严格的相似性/外推评价。

## 直觉解释

每个分子可以看成 fingerprint 空间中的一个点。相似分子通常更近，但二维图只是投影，会丢失信息。

化学上，这张图可以帮助我们问：疏水分子是否聚在一起？高分子量分子是否在某一区域？
logS 是否呈现连续变化？如果没有清晰区域，也不代表模型没用，可能只是二维投影不足以表达高维结构。

## 数学/化学定义

PCA 寻找方差最大的方向，把高维矩阵 `X` 投影到少数主成分：

```text
X_high_dim -> PC1, PC2
```

PCA 不知道化学规则，只根据数值方差工作。

## 准备 PCA 输入

这一格从 ESOL 中抽样，计算 Morgan fingerprint，再用 PCA 压缩到二维。
`explained_variance_ratio_` 表示 PC1/PC2 保留了多少数值方差信息；它通常不会很高，因为 fingerprint 很高维且稀疏。

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

sns.set_theme(style="whitegrid", context="notebook")


def mol_from_smiles(smiles):
    if pd.isna(smiles):
        return None
    return Chem.MolFromSmiles(str(smiles).strip())


def canonical_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def scaffold_smiles(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def descriptor_record(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "canonical_smiles": canonical_smiles(smiles),
        "scaffold": scaffold_smiles(mol),
    }


def build_esol_features():
    raw = pd.read_csv(RAW / "esol.csv")
    rows = []
    for row_id, row in raw.reset_index(drop=True).iterrows():
        desc = descriptor_record(row["smiles"])
        if desc is None:
            continue
        desc.update(
            {
                "row_id": row_id,
                "smiles": str(row["smiles"]).strip(),
                "logS": float(row["log solubility (mol/L)"]),
            }
        )
        rows.append(desc)
    return pd.DataFrame(rows)


def fingerprint_array(smiles, n_bits=1024):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    matrix = np.zeros((len(smiles), n_bits), dtype=np.int8)
    for idx, smi in enumerate(smiles):
        fp = generator.GetFingerprint(mol_from_smiles(smi))
        DataStructs.ConvertToNumpyArray(fp, matrix[idx])
    return matrix


DESCRIPTOR_COLUMNS = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "AromaticRings",
    "FractionCSP3",
    "HeavyAtomCount",
]

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

esol = build_esol_features()

# 为了运行快，只抽取最多 600 个分子；random_state 保证每次结果可复现。
sample = esol.sample(n=min(600, len(esol)), random_state=RANDOM_STATE).reset_index(drop=True)
fp = fingerprint_array(sample["smiles"], n_bits=384)

# PCA 只看 fingerprint 矩阵的数值方差，不知道 logS 或化学机制。
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(fp)

space = sample[["smiles", "logS", "MolWt", "MolLogP", "TPSA"]].copy()
space["PC1"] = coords[:, 0]
space["PC2"] = coords[:, 1]
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))
space.head()

## 代码

下面用颜色表示 logS。观察空间位置是否与溶解度、分子量或 LogP 有关系。

In [ ]:
# 如果颜色形成连续区域，说明二维投影捕捉到一部分和 logS 相关的结构差异。
plt.figure(figsize=(7, 5))
scatter = plt.scatter(space["PC1"], space["PC2"], c=space["logS"], cmap="coolwarm", s=24, alpha=0.8)
plt.colorbar(scatter, label="logS")
plt.title("PCA of Morgan fingerprints")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()

## 换 descriptor 给点上色

同一张二维坐标可以用不同化学量上色。

区分的概念：“空间位置由 fingerprint 决定”，而“颜色只是我们叠加上去帮助解释的属性”。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=space, x="PC1", y="PC2", hue="MolWt", palette="viridis", ax=axes[0], legend=False)
axes[0].set_title("Colored by MolWt")
sns.scatterplot(data=space, x="PC1", y="PC2", hue="MolLogP", palette="magma", ax=axes[1], legend=False)
axes[1].set_title("Colored by MolLogP")
plt.tight_layout()

## 看投影边缘的分子

这一格抽出 PC1/PC2 极端位置的分子。边缘点常常帮助解释 PCA 方向可能对应哪些结构差异，
但不要把 PC1/PC2 直接解释成单一化学性质。

In [ ]:
# 取 PC1/PC2 两端的分子，看看二维图边缘到底是什么结构。
extreme = pd.concat(
    [
        space.nsmallest(3, "PC1"),
        space.nlargest(3, "PC1"),
        space.nsmallest(3, "PC2"),
        space.nlargest(3, "PC2"),
    ]
).drop_duplicates().head(12)

Draw.MolsToGridImage(
    [mol_from_smiles(s) for s in extreme["smiles"]],
    legends=[f"logS={v:.1f}" for v in extreme["logS"]],
    molsPerRow=4,
    subImgSize=(230, 170),
)

## 观察问题

1. PCA 图上相邻分子一定化学相似吗？
2. logS 的颜色是否形成清晰区域？
3. 高维 fingerprint 投影到二维会丢失哪些信息？

### Hints

1. 不一定。二维邻近只说明在 PC1/PC2 上接近；高维 fingerprint 中的差异可能被压缩掉。
2. 如果颜色有梯度，说明某些结构差异和 logS 相关；如果颜色混杂，可能说明 logS 受多种因素影响或投影不足。
3. 会丢失局部结构 bit、稀有片段、非线性关系和高维邻居关系。二维图适合提出问题，不适合作为最终证据。

## 小结

分子空间图适合建立直觉、找异常点和讲故事，但不能替代严格评价。二维距离只是高维结构的一种压缩结果。